# Day 5 Capstone Project: AI Brochure Generator
We will build a tool that reads a website homepage, discovers its sub-pages, crawls them, and generates a sales brochure for prospective clients.

## Models Used:
- `llama3.2:1b` (Ollama)

In [1]:
import os
import json
import ollama
from IPython.display import Markdown, display, update_display

# import our custom scraping Function from scraper.py
from scraper import fetch_website_content,fetch_website_links

print("Setup complete. Scraper loaded!")

Setup complete. Scraper loaded!


In [2]:
# let see what links are found on the webpage.
target_url = "https://huggingface.co"

# Fetch all raw links from the page
raw_links = fetch_website_links(target_url)

print(f"Found {len(raw_links)} links on the website:")

print("-" * 50)

for link in raw_links[:15]:
    print(link)

Found 102 links on the website:
--------------------------------------------------
/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog
/posts


## Step 2: Use LLM to Filter Relevant Links
Many links on a page are external (like Twitter, LinkedIn) or irrelevant to a brochure (like Privacy Policy, Terms).

We will write a system prompt telling the LLM to act as a Link Selector. We will pass it the list of raw links and ask it to output a clean JSON object containing ONLY the links we need (about page, careers page, services page).

### Filter Links with Ollama

In [3]:
# System prompt that defines what Linkes are relevant and requests Json formate

link_system_prompt=""" 
You are provided with a list of link found on a webpage.
Decide which of these links would be most relevant to 
include in a sales broucher about the company.
Include pages link link About Us, Careers, Srvices, Projects, or Products.
Do not includes Terms of Service, Privacy Policy, or external 
social media links (Twitter, LinkedIn).

You must respond in JSON formate matching this example structure:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.url/careers"}
    ]
}
"""

user_prompt = f"""
Here is a list of links found on {target_url}.
Please select the relevant ones for a company brochure. If a 
link is relative (e.g. '/about'), convert it to an absolute 
URL (e.g. 'https://edwarddonner.com/about').

Links:
{chr(10).join(raw_links)}
"""

print("LLM is selecting relevant links...")

response= ollama.chat(
    model='llama3.2:1b',
    messages=[
        {'role': 'system', 'content': link_system_prompt},
        {'role': 'user', 'content': user_prompt}
    ],
    format='json' # Force Ollama to output valid JSON
)

# Parse and display the filtered links
filtered_data = json.loads(response.message.content)

print("\n Selected Relevant lLinks:")
print(json.dumps(filtered_data,indent=2))

LLM is selecting relevant links...

 Selected Relevant lLinks:
{
  "links": [
    {
      "type": "about page",
      "url": "https://huggingface.co/brand"
    },
    {
      "type": "enterprise",
      "url": "https://endpoints.huggingface.co"
    },
    {
      "type": "pricing#spaces",
      "url": "https://pricing.huggingface.co"
    }
  ]
}


### Scrape Relevant Pages and Combine Content

In [ ]:
combined_text = ""

# 1. Scrape the main homepage first
print(f"Scraping homepage: {target_url}")
combined_text += f"=== HOMEPAGE CONTENT ===\n"
combined_text += fetch_website_content(target_url) + "\n\n"

# print(combined_text)

# 2. Scrap each of the relevant sub-pages selected by the LLM

for item in filtered_data.get("links",[]):
    page_type = item.get("type")
    page_url = item.get("url")

    print(f"Scraping {page_type}: {page_url}")
    combined_text += f"=== {page_type.upper()} CONTENT ===\n"
    combined_text += fetch_website_content(page_url) + "\n\n"

print("\nWeb scraping complete!")
print(f"Total character size of collected data: {len(combined_text)}")

Scraping homepage: https://huggingface.co
about page - https://huggingface.co/brand
Scraping about page: https://huggingface.co/brand
enterprise - https://endpoints.huggingface.co
Scraping enterprise: https://endpoints.huggingface.co
pricing#spaces - https://pricing.huggingface.co
Scraping pricing#spaces: https://pricing.huggingface.co
Error fetching content from https://pricing.huggingface.co: HTTPSConnectionPool(host='pricing.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000296869A7B10>: Failed to resolve 'pricing.huggingface.co' ([Errno 11001] getaddrinfo failed)"))

Web scraping complete!
Total character size of collected data: 5078


## Step 3: Generate the Final Brochure
Now that we have collected the text from all relevant pages, we will feed this text into the LLM and ask it to write a structured corporate brochure.

### Generate Brochure

In [ ]:
# System instructions defining brochure sections and tone
brochure_system_prompt = """ 
You are an expert copywriter. Write a professional, engaging 
sales brochure for the company based on the text provided.
The brochure should have the following sections:
1. Company Overview (Who they are and their mission)
2. Key Services / Products
3. Careers & Culture (Why work here)
4. Contact Information

Make it sound exciting for prospective clients and 
candidates. Format the output with clear markdown headings.
"""

print("Generating brochure. Please wait...")
response_brochure = ollama.chat(
    model="llama3.2:1b",
    messages=[
        {"role": "system", "content": brochure_system_prompt},
        {
            "role": "user",
            "content": f"Here is the text content from the website:\n\n{combined_text}",
        },
    ],
    stream=True
)

# print("\n FINAL GENERATED BROCHURE")
# print("="*80)
# print(response_brochure.message.content)

""" 
With a small adjustment, we can change this so that the results stream back from AI,
with the familiar typewriter animation
"""

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in response_brochure:
    response += chunk.message.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)

Generating brochure. Please wait...


**Hugging Face: The AI Community Building the Future**

**[Cover Page]**

Unlock the Power of Machine Learning with Hugging Face
------------------------

Hugging Face is the premier platform for machine learning experts, researchers, and innovators to collaborate, share knowledge, and build applications. Our collaborative environment empowers you to explore new ideas, discover trending models, and accelerate your AI journey.

**[Company Overview]**

At Hugging Face, we're building a community that's shaping the future of artificial intelligence. As a leader in open-source machine learning, we provide a comprehensive platform for:

* **Unlimited public models**: Explore 2M+ models, including those from top research institutions
* **Trending datasets**: Discover and collaborate on high-impact datasets, such as Glint-Research/Fable-5-traces
* **Collaborative development**: Share, discuss, and build applications with a global community of experts

**[Key Services]**

Explore Our Range of AI Solutions

* **Inference Endpoints**: Leverage our curated set of inference endpoints for efficient and scalable AI applications
* **OpenMythos**: Discover and deploy Open Source Cyber Security Agents that protect your organization from emerging threats
* **LocalLaws/LOCUS-v1**: Utilize our 500k+ datasets to fine-tune I2V models with identity & voice conditioning

**[Careers & Culture]**

Why Work at Hugging Face?

Join a Community of Ambitious Professionals
-------------------------------------------

Hugging Face is more than just a platform - it's a community built on open-source values, collaboration, and innovation. Our team members are:

* **Ambitious**: Passionate about machine learning and committed to shaping the future of AI
* **Innovative**: Leading experts in their fields, always looking for new ways to improve our platform
* **Collaborative**: Dedicated to working together with you to achieve success

**[Contact Information]**

Get in Touch with Us
--------------------

Learn more about our platform, join our community, or get in touch with our team. Contact us at:

Email: [huggingface.com/contact](mailto:huggingface.com/contact)
Phone: +1 (555) 123-4567
Website: huggingface.com

**[About Page]**

Our Mission and Vision
----------------------

At Hugging Face, we're on a mission to build a world where machine learning is accessible to everyone. Our vision is to create a collaborative platform that empowers developers, researchers, and innovators to shape the future of AI.

Join us in our quest to:

* **Empower diversity**: Foster an inclusive community that celebrates individuality and creativity
* **Accelerate innovation**: Support cutting-edge research and development in machine learning
* **Build a better world**: Collaborate on applications that benefit humanity and the environment

**[Inference Endpoints]**

Unlock the Power of Inference Endpoints
-----------------------------------------

Our curated set of inference endpoints provides:

* **Efficient models**: Leverage pre-trained models with high accuracy and speed
* **Scalable deployments**: Deploy models on any platform, from cloud to edge
* **Collaborative development**: Share and discuss model ideas with our community

Download our Inference Endpoints API documentation: [huggingface.com/endpoints](https://huggingface.com/endpoints)

**[OpenMythos]**

Discover and Deploy Open Source Cyber Security Agents
--------------------------------------------------------

Our OpenSource Cyber Security Agent is designed to protect your organization from emerging threats. Join our community of experts and learn how to:

* **Fine-tune I2V models**: Unlock the full potential of your security capabilities
* **Develop custom agents**: Collaborate with our team to create tailored security solutions

**[LocalLaws/LOCUS-v1]**

Utilize Our 500k+ Datasets for Fine-Tuning I2V Models
---------------------------------------------------------

Our datasets are designed to help you fine-tune I2V models with identity & voice conditioning. Explore:

* **Glint-Research/Fable-5-traces**: Discover trending models and applications
* **LocalLaws/LOCUS-v1**: Use our 500k+ datasets for high-quality training data

**[Log In]**

Login to our platform today and start exploring the latest AI trends, models, and applications.